In [ ]:
# @title 00 . session { display-mode: "form" }
# Environment check + shared helpers. Run once per runtime, before anything else.
import getpass, hashlib, json, os, re, shutil, signal, subprocess, sys, threading, time, uuid
import warnings
from pathlib import Path

import psutil
import requests
import torch
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

GIB = 1024 ** 3
CONTENT = Path("/content")
COMFY_DIR = CONTENT / "ComfyUI"
CUSTOM_NODES = COMFY_DIR / "custom_nodes"
WORK = CONTENT / "h3lab"
MODELS = COMFY_DIR / "models"
OUTPUT_DIR = COMFY_DIR / "output"
COMFY_LOG = WORK / "comfy.log"
WHEEL_DIR = CONTENT / "_wheels"

GGUF_DIR = CUSTOM_NODES / "ComfyUI-GGUF"
TURBO_DIR = CUSTOM_NODES / "ComfyUI-MiniMax-H3-Turbo"
CONTROL_DIR = CUSTOM_NODES / "H3LabControl"
FP16SAFE_DIR = CUSTOM_NODES / "ComfyUI-MiniMaxH3-FP16Safe"
TESPEED_DIR = CUSTOM_NODES / "TE-Speed-MiniMaxH3-MChip"
SPECTRUM_DIR = CUSTOM_NODES / "ComfyUI-Spectrum-MiniMax-H3"
FP16_FIX_FILE = CUSTOM_NODES / "minimax_h3_fp16_fix.py"

BASE_URL = "http://127.0.0.1:8188"

WORK.mkdir(parents=True, exist_ok=True)
WHEEL_DIR.mkdir(parents=True, exist_ok=True)

H3 = globals().setdefault("H3", {})


def _box_html(title, rows=None, kind="info", note="", details=""):
    colors = {"ok": "#0d7a3f", "warn": "#9a6b00", "err": "#b3261e",
              "info": "#1a5fb4", "idle": "#5f6368", "run": "#1a5fb4"}
    color = colors.get(kind, colors["info"])
    body = ""
    if rows:
        body += '<table style="border-collapse:collapse;width:100%;font:12px/1.55 monospace">'
        for label, value in rows:
            body += ('<tr><td style="padding:2px 12px 2px 0;color:#5f6368;'
                     'white-space:nowrap;vertical-align:top">' + str(label) + '</td>'
                     '<td style="padding:2px 0;word-break:break-word">' + str(value)
                     + '</td></tr>')
        body += "</table>"
    if note:
        body += ('<div style="margin-top:6px;color:#5f6368;font:12px/1.5 monospace">'
                 + note + "</div>")
    if details:
        body += ('<details style="margin-top:6px"><summary style="cursor:pointer;'
                 'color:#5f6368;font:12px monospace">log tail</summary>'
                 '<pre style="max-height:340px;overflow:auto;background:#f6f8fa;'
                 'padding:8px;font:11px/1.45 monospace;margin:6px 0 0">'
                 + details + '</pre></details>')
    return ('<div style="border:1px solid ' + color + ';border-radius:8px;margin:4px 0;'
            'overflow:hidden">'
            '<div style="background:' + color + ';color:#fff;padding:6px 12px;'
            'font:600 12px/1.4 monospace">' + title + '</div>'
            '<div style="padding:8px 12px">' + body + '</div></div>')


def box(title, rows=None, kind="info", note="", details=""):
    display(HTML(_box_html(title, rows, kind, note, details)))


class LiveBox:
    """Box that appears immediately and updates in place while work runs."""

    def __init__(self, key, title, rows=None, kind="run", note=""):
        self.display_id = "h3live-%s-%s" % (key, uuid.uuid4().hex[:6])
        self.title, self.rows = title, list(rows or [])
        self.kind, self.note, self.details = kind, note, ""
        self._render(first=True)

    def _render(self, first=False):
        try:
            display(HTML(_box_html(self.title, self.rows, self.kind, self.note,
                                   self.details)),
                    display_id=self.display_id, update=not first)
        except Exception:
            pass

    def set(self, title=None, rows=None, kind=None, note=None, details=None):
        if title is not None:
            self.title = title
        if rows is not None:
            self.rows = list(rows)
        if kind is not None:
            self.kind = kind
        if note is not None:
            self.note = note
        if details is not None:
            self.details = details
        self._render()

    def finish(self, title, rows=None, kind="ok", note="", details=""):
        self.set(title=title, rows=rows, kind=kind, note=note, details=details)


class Ticker:
    """Refresh a LiveBox every interval seconds via a build(live) callback."""

    def __init__(self, live, build, interval=2.5):
        self.live, self.build = live, build
        self.interval = max(0.5, float(interval))
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self.stop.wait(self.interval):
            try:
                self.build(self.live)
            except Exception:
                pass

    def start(self):
        try:
            self.build(self.live)
        except Exception:
            pass
        self.thread.start()
        return self

    def stop_now(self):
        self.stop.set()


def fmt_secs(seconds):
    seconds = int(max(0, seconds))
    if seconds < 60:
        return "%ds" % seconds
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return "%dm %02ds" % (minutes, sec)
    hours, minutes = divmod(minutes, 60)
    return "%dh %02dm" % (hours, minutes)


def pbar(fraction, width=24):
    try:
        fraction = float(fraction)
    except Exception:
        fraction = 0.0
    fraction = min(1.0, max(0.0, fraction))
    filled = int(round(fraction * width))
    return "[" + "#" * filled + "-" * (width - filled) + "] %3d%%" % int(round(fraction * 100))


def gb(nbytes):
    return "%.2f GiB" % (nbytes / GIB)


def sh(cmd, cwd=None, timeout=3600):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, timeout=timeout,
                       env=env)
    return p.returncode, (p.stdout or "") + (p.stderr or "")


def tail_comfy_log(offset=0, limit=120):
    if not COMFY_LOG.exists():
        return ""
    text = COMFY_LOG.read_bytes()[offset:].decode("utf-8", errors="replace")
    lines = [l for l in text.splitlines() if l.strip()]
    return "\n".join(lines[-limit:])


ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")
TQDM_RE = re.compile(
    r"(\d+)%\|[^|\n]*\|\s*(\d+)/(\d+)\s*\[([0-9:]+)<([0-9:]+|\?)"
    r"(?:,\s*([0-9.]+)(s/it|it/s))?")


def parse_clock(value):
    total = 0.0
    for part in value.split(":"):
        total = total * 60 + float(part)
    return total


def latest_progress(text):
    matches = list(TQDM_RE.finditer(ANSI_RE.sub("", text).replace("\r", "\n")))
    if not matches:
        return None
    percent, done, total, elapsed, _remaining, rate, unit = matches[-1].groups()
    done, elapsed_s = int(done), parse_clock(elapsed)
    return {"percent": int(percent), "steps_completed": done, "steps_total": int(total),
            "elapsed_in_loop_s": elapsed_s,
            "observed_s_per_step": (elapsed_s / done) if done else None,
            "reported_s_per_step": (float(rate) if unit == "s/it"
                                    else (1.0 / float(rate) if rate and float(rate) else None))
            if rate else None}


def stop_comfy():
    proc, handle = H3.get("proc"), H3.get("log_handle")
    if proc and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=20)
        except Exception:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait(timeout=10)
            except Exception:
                pass
    if handle:
        try:
            handle.close()
        except Exception:
            pass
    H3["proc"], H3["log_handle"] = None, None


def start_comfy(dtype_flag, wait_s=1200, extra=()):
    """Start ComfyUI fresh. Returns (system_stats, log_offset_at_start)."""
    stop_comfy()
    handle = COMFY_LOG.open("ab")
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--disable-auto-launch", "--preview-method", "none", "--cache-classic"]
    if dtype_flag:
        cmd.append(dtype_flag)
    cmd += list(extra)
    proc = subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=handle,
                            stderr=subprocess.STDOUT, start_new_session=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    H3["proc"], H3["log_handle"] = proc, handle
    offset = handle.tell()
    started = time.time()
    while time.time() - started < wait_s:
        if proc.poll() is not None:
            tail = tail_comfy_log(max(0, offset - 40000), 30)
            stop_comfy()
            raise RuntimeError("ComfyUI exited during startup:\n" + tail)
        try:
            r = requests.get(BASE_URL + "/system_stats", timeout=3)
            if r.status_code == 200:
                return r.json(), offset
        except Exception:
            pass
        time.sleep(2)
    stop_comfy()
    raise RuntimeError("ComfyUI did not answer /system_stats within %ds" % wait_s)


def get_object_info():
    return requests.get(BASE_URL + "/object_info", timeout=600).json()


def server_up():
    try:
        return requests.get(BASE_URL + "/system_stats", timeout=3).status_code == 200
    except Exception:
        return False


def validate_graph(graph, object_info):
    errors = []
    for node_id, spec in graph.items():
        info = object_info.get(spec["class_type"])
        if info is None:
            errors.append("%s: unknown class %s" % (node_id, spec["class_type"]))
            continue
        schema = info.get("input", {})
        allowed = (set(schema.get("required", {})) | set(schema.get("optional", {}))
                   | set(schema.get("hidden", {})))
        supplied = set(spec["inputs"])
        missing = set(schema.get("required", {})) - supplied
        if missing:
            errors.append("%s/%s missing %s" % (node_id, spec["class_type"], sorted(missing)))
        unknown = supplied - allowed
        if unknown:
            errors.append("%s/%s unknown %s" % (node_id, spec["class_type"], sorted(unknown)))
    return errors


_host = psutil.virtual_memory()
_disk = shutil.disk_usage(str(CONTENT))
box("00 . session ready", [
    ("python", sys.version.split()[0]),
    ("torch", "%s  (cuda %s)" % (torch.__version__, torch.version.cuda)),
    ("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"),
    ("host RAM", "%.1f GiB total / %.1f GiB free" % (_host.total / GIB, _host.available / GIB)),
    ("/content free", gb(_disk.free)),
], kind="ok",
   note="Next: 01 base stack, 02 models, then 03 launches the H3 Lab Studio (the UI has all "
        "run config, methods, encode/sample/decode and logs).")


In [ ]:
# @title 01 . base stack { display-mode: "form" }
COMFY_REF = "0.30.0 (known good)" #@param ["0.30.0 (known good)", "0.41.0 (latest)"]

COMFY_REFS = {
    "0.30.0 (known good)": "9a9fdb10ed144ce760d9682cb247526ea23cc525",
    "0.41.0 (latest)": "v0.41.0",
}
GGUF_COMMIT = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"
TURBO_COMMIT = "55fee864dd7b2976b1c4ce3c3d5f7968f181409f"
COMFY_CHECKOUT = COMFY_REFS[COMFY_REF]
BASELINE = COMFY_CHECKOUT == COMFY_REFS["0.30.0 (known good)"]

if "H3" not in globals():
    raise RuntimeError("run cell 00 first")

import os, re, sys, time

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")

def git_sync(url, dest, ref):
    if not (dest / ".git").is_dir():
        if dest.exists():
            raise RuntimeError("path exists and is not a git repo: %s" % dest)
        rc, out = sh(["git", "clone", url, str(dest)], timeout=1800)
        if rc:
            raise RuntimeError("clone failed for %s:\n%s" % (url, out[-600:]))
    rc, _ = sh(["git", "-C", str(dest), "cat-file", "-e", ref + "^{commit}"])
    if rc:
        rc, out = sh(["git", "-C", str(dest), "fetch", "origin", ref, "--depth", "1"],
                     timeout=1800)
        if rc:
            rc, out = sh(["git", "-C", str(dest), "fetch", "origin"], timeout=1800)
            if rc:
                raise RuntimeError("fetch failed for %s:\n%s" % (dest, out[-600:]))
    rc, out = sh(["git", "-C", str(dest), "checkout", "-f", ref])
    if rc:
        if re.fullmatch(r"[0-9a-f]{40}", ref):
            raise RuntimeError("checkout failed for %s (%s):\n%s" % (dest, ref, out[-600:]))
        rc, out = sh(["git", "-C", str(dest), "checkout", "-f", "FETCH_HEAD"])
        if rc:
            raise RuntimeError("checkout failed for %s (%s):\n%s" % (dest, ref, out[-600:]))
    return sh(["git", "-C", str(dest), "rev-parse", "HEAD"])[1].strip()

rows, errors = [], []

live = LiveBox("01", "01 . base stack - running", kind="run")
stage = {"label": "updating repositories"}
_started_01 = time.perf_counter()

def _tick01(live):
    live.set(rows=list(rows) + [("elapsed", fmt_secs(time.perf_counter() - _started_01))],
             note=stage["label"] + " ...")

ticker = Ticker(live, _tick01, 2.0).start()

try:
    stage["label"] = "updating ComfyUI repository"
    head = git_sync("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, COMFY_CHECKOUT)
    expect = COMFY_CHECKOUT if re.fullmatch(r"[0-9a-f]{40}", COMFY_CHECKOUT) else None
    if expect and head != expect:
        raise RuntimeError("ComfyUI HEAD is %s but %s was requested; refusing to run a "
                           "different version silently" % (head[:12], expect[:12]))
    rows.append(("ComfyUI", head[:12] + (" (pinned)" if expect else " (ref %s)" % COMFY_CHECKOUT)))
except Exception as exc:
    errors.append("ComfyUI: %s" % exc)

try:
    stage["label"] = "installing ComfyUI requirements (pip)"
    req = COMFY_DIR / "requirements.txt"
    keep, count = [], 0
    for line in req.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith("#"):
            name = re.split(r"[<>=!~ ]", stripped, maxsplit=1)[0].strip().lower()
            if name in ("torch", "torchvision", "torchaudio"):
                continue
            count += 1
        keep.append(line)
    filtered = WORK / "comfy_requirements.txt"
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
                 timeout=2400)
    if rc:
        raise RuntimeError("pip install -r requirements failed:\n" + out[-800:])
    rows.append(("python deps", "%d packages from requirements.txt (torch kept as Colab's)" % count))
except Exception as exc:
    errors.append("ComfyUI deps: %s" % exc)

try:
    stage["label"] = "updating ComfyUI-GGUF"
    head = git_sync("https://github.com/city96/ComfyUI-GGUF.git", GGUF_DIR, GGUF_COMMIT)
    loader = GGUF_DIR / "loader.py"
    text = loader.read_text(encoding="utf-8")
    anchor = '"qwen_image"}'
    patched = '"qwen_image", "minimax_h3"}'
    if patched in text:
        state = "already patched"
    elif text.count(anchor) == 1:
        loader.write_text(text.replace(anchor, patched, 1), encoding="utf-8")
        state = "patched IMG_ARCH_LIST"
    else:
        raise RuntimeError("arch anchor appears %d times in loader.py (expected 1); "
                           "inspect IMG_ARCH_LIST by hand" % text.count(anchor))
    rows.append(("ComfyUI-GGUF", head[:12] + ", " + state))
except Exception as exc:
    errors.append("ComfyUI-GGUF: %s" % exc)

try:
    stage["label"] = "updating Turbo nodes"
    head = git_sync("https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git", TURBO_DIR,
                    TURBO_COMMIT)
    grid = TURBO_DIR / "h3_silu_temb_grid.safetensors"
    if not grid.is_file():
        raise RuntimeError("h3_silu_temb_grid.safetensors missing from the Turbo checkout; "
                           "the LoRA would apply only partially and silently")
    rows.append(("Turbo nodes", head[:12] + ", silu grid %d bytes" % grid.stat().st_size))
except Exception as exc:
    errors.append("Turbo nodes: %s" % exc)

try:
    stage["label"] = "installing custom node requirements"
    node_deps = "no requirements.txt found"
    dep_rows = []
    for label, d in (("GGUF", GGUF_DIR), ("Turbo", TURBO_DIR),
                     ("TE-Speed", TESPEED_DIR), ("Spectrum", SPECTRUM_DIR),
                     ("FP16Safe", FP16SAFE_DIR)):
        req = d / "requirements.txt"
        if d.is_dir() and req.is_file():
            rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
                         timeout=1200)
            if rc:
                raise RuntimeError("%s requirements failed:\n%s" % (label, out[-500:]))
            dep_rows.append(label)
    if dep_rows:
        node_deps = ", ".join(dep_rows)
    rows.append(("node deps", node_deps))
except Exception as exc:
    errors.append("custom node deps: %s" % exc)

PACK_SRC = r'''"""H3Lab control nodes: conditioning/latent persistence + optional patches.

Three-stage split support (encode -> sample -> decode), plus:
  - H3Fp32TextBranch: partial fp16 overflow fix (condition_proj input upcast)
  - H3SagePatch: attention override for SageAttention (adapted from KJNodes'
    PathchSageAttentionKJ, MIT, trimmed to the sm75-safe kernel)
"""
import json
import logging
from pathlib import Path

import torch

H3_TORCH_NESTED = "__h3_torch_nested__"
H3_COMFY_NESTED = "__h3_comfy_nested__"


def _is_comfy_nested(obj):
    """comfy.nested_tensor.NestedTensor, duck-typed."""
    return (not isinstance(obj, torch.Tensor)
            and bool(getattr(obj, "is_nested", False))
            and hasattr(obj, "unbind"))


def _restore(obj):
    if isinstance(obj, dict):
        keys = set(obj.keys())
        if keys == {H3_TORCH_NESTED}:
            return torch.nested.nested_tensor(list(obj[H3_TORCH_NESTED]))
        if keys == {H3_COMFY_NESTED}:
            from comfy.nested_tensor import NestedTensor
            return NestedTensor(list(obj[H3_COMFY_NESTED]))
        return {key: _restore(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_restore(value) for value in obj)
    if isinstance(obj, list):
        return [_restore(value) for value in obj]
    return obj


def _summarize(obj):
    if _is_comfy_nested(obj):
        return {"kind": "comfy_nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {"kind": "nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
        return {"kind": "tensor", "shape": list(obj.shape), "dtype": str(obj.dtype),
                "device": str(obj.device),
                "finite": bool(torch.isfinite(obj).all().item())
                if obj.is_floating_point() else None}
    if isinstance(obj, dict):
        return {key: _summarize(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_summarize(value) for value in obj]
    return {"kind": type(obj).__name__, "repr": repr(obj)[:200]}


def _detach_to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {H3_TORCH_NESTED: [p.detach().cpu() for p in obj.unbind()]}
        return obj.detach().cpu()
    if _is_comfy_nested(obj):
        return {H3_COMFY_NESTED: [p.detach().cpu() for p in obj.unbind()]}
    if isinstance(obj, dict):
        return {key: _detach_to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_detach_to_cpu(value) for value in obj)
    if isinstance(obj, list):
        return [_detach_to_cpu(value) for value in obj]
    return obj


class H3SaveConditioning:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "conditioning": ("CONDITIONING",), "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ()
    FUNCTION = "save"
    CATEGORY = "H3Lab"
    OUTPUT_NODE = True

    def save(self, conditioning, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = {"schema_version": 1,
                   "conditioning": _detach_to_cpu(conditioning),
                   "samples": _detach_to_cpu(samples)}
        destination = directory / (key + ".pt")
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        (directory / (key + ".summary.json")).write_text(json.dumps(
            {"key": key, "bytes": destination.stat().st_size,
             "conditioning": _summarize(payload["conditioning"]),
             "samples": _summarize(payload["samples"])}, indent=2), encoding="utf-8")
        logging.info("H3_SAVE_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return {}


class H3LoadConditioning:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ("CONDITIONING", "LATENT")
    RETURN_NAMES = ("conditioning", "samples")
    FUNCTION = "load"
    CATEGORY = "H3Lab"

    def load(self, store_dir, key):
        source = Path(store_dir) / (key + ".pt")
        if not source.is_file():
            raise RuntimeError("Precomputed conditioning is missing: %s" % source)
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError("Unsupported schema: %s" % payload.get("schema_version"))
        logging.info("H3_LOAD_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": source.stat().st_size}, separators=(",", ":")))
        return (_restore(payload["conditioning"]), _restore(payload["samples"]))


class H3SaveLatent:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save"
    CATEGORY = "H3Lab"
    OUTPUT_NODE = True

    def save(self, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        path = directory / (key + ".pt")
        torch.save({"schema_version": 1, "key": key,
                    "samples": _detach_to_cpu(samples)}, path)
        logging.info("H3_SAVE_LATENT %s", json.dumps(
            {"key": key, "path": str(path), "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3LoadLatent:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "load"
    CATEGORY = "H3Lab"

    def load(self, store_dir, key):
        path = Path(store_dir) / (key + ".pt")
        if not path.is_file():
            raise RuntimeError("No latent at %s. Run the sampling cell first." % path)
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError("Unsupported schema: %s" % payload.get("schema_version"))
        samples = _restore(payload["samples"])
        logging.info("H3_LOAD_LATENT %s", json.dumps(
            {"key": key, "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3LatentProbe:
    """Report the sampled latent. Raises on non-finite values, after saving the report."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "filename_prefix": ("STRING", {"default": "h3_lab/probe"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save_summary"
    CATEGORY = "H3Lab"
    OUTPUT_NODE = True

    def save_summary(self, samples, filename_prefix):
        import folder_paths
        summary = _summarize(samples)
        packed = samples.get("samples")
        parts = packed.unbind() if (_is_comfy_nested(packed)
                                    or getattr(packed, "is_nested", False)) else [packed]
        finite = [bool(torch.isfinite(p).all().item()) for p in parts
                  if isinstance(p, torch.Tensor)]
        output_path = (Path(folder_paths.get_output_directory())
                       / (filename_prefix + ".latent_summary.json"))
        output_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {"summary": summary,
                   "shapes": [list(p.shape) for p in parts if isinstance(p, torch.Tensor)],
                   "finite": finite}
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        logging.info("H3_LATENT_PROBE %s", json.dumps(payload["shapes"], separators=(",", ":")))
        if not finite or not all(finite):
            raise RuntimeError("Sampled latent contains non-finite values: %s" % summary)
        return (samples,)


class H3Fp32TextBranch:
    """Run the text branch in fp32 (condition_proj input absmax ~15424 in fp16)."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"model": ("MODEL",)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "apply"
    CATEGORY = "H3Lab"

    def apply(self, model):
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        proj = getattr(target, "condition_proj", None)
        if proj is None:
            logging.warning("H3_FP32_TEXT_BRANCH no condition_proj found; nothing patched")
            return (model,)
        if getattr(proj, "_h3_fp32_text_branch", False):
            logging.info("H3_FP32_TEXT_BRANCH already_applied")
            return (model,)
        original = proj.forward

        def forward(x, *args, **kwargs):
            if isinstance(x, torch.Tensor) and x.dtype == torch.float16:
                x = x.to(torch.float32)
            return original(x, *args, **kwargs)

        proj.forward = forward
        proj._h3_fp32_text_branch = True
        logging.info("H3_FP32_TEXT_BRANCH applied to condition_proj")
        return (model,)


class H3SagePatch:
    """Route model attention through SageAttention (int8 QK, fp16 PV)."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "model": ("MODEL",),
            "mode": (["sageattn_qk_int8_pv_fp16_cuda", "auto"],)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "patch"
    CATEGORY = "H3Lab"

    def patch(self, model, mode):
        import torch
        from sageattention import sageattn, sageattn_qk_int8_pv_fp16_cuda
        from comfy.ldm.modules.attention import attention_pytorch
        try:
            from comfy.ldm.modules.attention import wrap_attn
        except Exception:
            def wrap_attn(f):
                f.__wrapped__ = f
                return f

        if mode == "auto":
            def kernel(q, k, v, is_causal=False, attn_mask=None, tensor_layout="NHD"):
                return sageattn(q, k, v, is_causal=is_causal, attn_mask=attn_mask,
                                tensor_layout=tensor_layout)
        else:
            def kernel(q, k, v, is_causal=False, attn_mask=None, tensor_layout="NHD"):
                return sageattn_qk_int8_pv_fp16_cuda(
                    q, k, v, is_causal=is_causal, attn_mask=attn_mask,
                    pv_accum_dtype="fp32", tensor_layout=tensor_layout)

        kernel = torch.compiler.disable()(kernel)

        @wrap_attn
        def attention_sage(q, k, v, heads, mask=None, attn_precision=None,
                           skip_reshape=False, skip_output_reshape=False, **kwargs):
            if kwargs.get("low_precision_attention", True) is False:
                return attention_pytorch(q, k, v, heads, mask=mask,
                                         skip_reshape=skip_reshape,
                                         skip_output_reshape=skip_output_reshape, **kwargs)
            in_dtype = v.dtype
            if (q.dtype == torch.float32 or k.dtype == torch.float32
                    or v.dtype == torch.float32):
                q, k, v = q.to(torch.float16), k.to(torch.float16), v.to(torch.float16)
            if skip_reshape:
                b, _, _, dim_head = q.shape
                tensor_layout = "HND"
            else:
                b, _, dim_head = q.shape
                dim_head //= heads
                q, k, v = map(lambda t: t.view(b, -1, heads, dim_head), (q, k, v))
                tensor_layout = "NHD"
            if mask is not None:
                if mask.ndim == 2:
                    mask = mask.unsqueeze(0)
                if mask.ndim == 3:
                    mask = mask.unsqueeze(1)
            seq_dim = 2 if tensor_layout == "HND" else 1
            if any((t.shape[seq_dim] - 1) * t.stride(seq_dim) >= 2 ** 31 for t in (q, k, v)):
                q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
            out = kernel(q, k, v, attn_mask=mask, is_causal=False,
                         tensor_layout=tensor_layout).to(in_dtype)
            if tensor_layout == "HND":
                if not skip_output_reshape:
                    out = out.transpose(1, 2).reshape(b, -1, heads * dim_head)
            else:
                if skip_output_reshape:
                    out = out.transpose(1, 2)
                else:
                    out = out.reshape(b, -1, heads * dim_head)
            return out

        clone = model.clone()
        clone.model_options["transformer_options"]["optimized_attention_override"] = (
            lambda func, *args, **kwargs: attention_sage.__wrapped__(*args, **kwargs))
        logging.info("[H3Lab] sage attention override active (mode=%s)", mode)
        return (clone,)


NODE_CLASS_MAPPINGS = {
    "H3SaveConditioning": H3SaveConditioning,
    "H3LoadConditioning": H3LoadConditioning,
    "H3SaveLatent": H3SaveLatent,
    "H3LoadLatent": H3LoadLatent,
    "H3LatentProbe": H3LatentProbe,
    "H3Fp32TextBranch": H3Fp32TextBranch,
    "H3SagePatch": H3SagePatch,
}

NODE_DISPLAY_NAME_MAPPINGS = {
    "H3SaveConditioning": "H3 Save Conditioning",
    "H3LoadConditioning": "H3 Load Conditioning",
    "H3SaveLatent": "H3 Save Latent",
    "H3LoadLatent": "H3 Load Latent",
    "H3LatentProbe": "H3 Latent Probe",
    "H3Fp32TextBranch": "H3 Fp32 Text Branch",
    "H3SagePatch": "H3 Sage Attention Patch",
}
'''

try:
    stage["label"] = "writing H3Lab control nodes"
    CONTROL_DIR.mkdir(parents=True, exist_ok=True)
    (CONTROL_DIR / "__init__.py").write_text(PACK_SRC, encoding="utf-8")
    rc, out = sh([sys.executable, "-c",
                  "import py_compile; py_compile.compile(%r, doraise=True)"
                  % str(CONTROL_DIR / "__init__.py")])
    if rc:
        raise RuntimeError("control pack failed to compile:\n" + out[-600:])
    rows.append(("control pack", "7 nodes written + compiled"))
except Exception as exc:
    errors.append("control pack: %s" % exc)

try:
    stage["label"] = "checking av"
    rc, out = sh([sys.executable, "-c", "import av; print(av.__version__)"])
    if rc:
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "av"], timeout=900)
        rc2, ver = sh([sys.executable, "-c", "import av; print(av.__version__)"])
        if rc or rc2:
            raise RuntimeError("av install failed:\n" + (out + ver)[-600:])
        rows.append(("av", ver.strip() + " (installed)"))
    else:
        rows.append(("av", out.strip()))
except Exception as exc:
    errors.append("av: %s" % exc)

ticker.stop_now()
version_note = ""
if not BASELINE:
    version_note = ("<br>Non-baseline ComfyUI (%s): the Studio's fp16 fix is tested only "
                    "against v0.30.0; if sampling reports NaN or 'fp16 did not take effect', "
                    "switch back. TE-Speed/Spectrum V3 nodes are likelier to load here."
                    % COMFY_CHECKOUT)
live.finish("01 . base stack " + ("FAILED" if errors else "installed"), rows=rows,
            kind="err" if errors else ("warn" if version_note else "ok"),
            note=(("<br>".join(errors) + "<br>Fix the offending piece and rerun this cell.")
                  if errors else "Pinned commits + Python requirements (torch untouched) + "
                  "guarded GGUF arch patch + control nodes. Method repos install from the "
                  "Studio's setup tab." + version_note))
if errors:
    raise RuntimeError("; ".join(errors))


In [ ]:
# @title 02 . models { display-mode: "form" }
DIT_CHOICE = "abiray_Q3_K_M" #@param ["abiray_Q3_K_M", "abiray_Q4_K_M"]
INCLUDE_TEXT_ENCODER = "auto" #@param ["auto", "yes", "no"]
TURBO_LORAS = "v4 only" #@param ["v4 only", "v4 + v1", "none"]
FORCE_REDOWNLOAD = False #@param {type:"boolean"}

if "H3" not in globals():
    raise RuntimeError("run cell 00 first")
if not (COMFY_DIR / ".git").is_dir():
    raise RuntimeError("ComfyUI is not installed yet; run cell 01 first")

import os, time
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")

from pathlib import Path

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import hf_hub_download

try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

MODEL_REPO = "Comfy-Org/MiniMax-H3"
MODEL_REVISION = "0543966fbdce5ba05709a8f2031c94bdba629b4a"
DIT_CHOICES = {
    "abiray_Q3_K_M": {
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF",
        "file": "MiniMax-H3-FL2VA-Pruned-Q3_K_M.gguf",
        "bytes": 8902845536,
        "sha256": "c844d70dbb4df67b7d9605c669bff35d9c631bd0507143cf0276f984de32247d"},
    "abiray_Q4_K_M": {
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF",
        "file": "MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf",
        "bytes": 11564180576,
        "sha256": "d74e644906fe95b8c7bf4837ba9d3f9392925f7301675a0993d419b62a4824a3"},
}
TURBO_LORA_CHOICES = {
    "v4_step600_ema": {
        "repo": "larryvrh/MiniMax-H3-Turbo-Lora",
        "file": "minimax_h3_turbo_v4_step600_ema.safetensors",
        "bytes": 779849816,
        "sha256": "5f3a626cd72c93a8b9318d6760c510bc5092d2ab13aaba1f932c5bab07a416d3"},
    "v1_850_ema": {
        "repo": "larryvrh/MiniMax-H3-Turbo-Lora",
        "file": "minimax_h3_turbo_4step_ema_ckpt850.safetensors",
        "bytes": 779849816,
        "sha256": "5a6eeba171cf183020a4ad48774bb2968f29f8168afd6ec17a04987f3528b4ea"},
}
TE_FILE = {"path": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
           "name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
           "bytes": 15687142551,
           "sha256": "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6"}
VAE_FILES = {
    "video": {"path": "vae/minimax_h3_video_vae_fp16.safetensors",
              "name": "minimax_h3_video_vae_fp16.safetensors",
              "bytes": 5207808496,
              "sha256": "7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522"},
    "audio": {"path": "vae/minimax_h3_audio_vae_fp32.safetensors",
              "name": "minimax_h3_audio_vae_fp32.safetensors",
              "bytes": 605254808,
              "sha256": "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48"},
}

for sub in ("diffusion_models", "loras", "vae", "text_encoders"):
    (MODELS / sub).mkdir(parents=True, exist_ok=True)

if INCLUDE_TEXT_ENCODER == "yes":
    need_te, te_reason = True, "forced by config"
elif INCLUDE_TEXT_ENCODER == "no":
    need_te, te_reason = False, "disabled by config"
else:
    cond_dir = WORK / "conditioning"
    cached = [p for p in cond_dir.glob("*.pt")
              if p.is_file() and p.stat().st_size > 1024 * 1024] if cond_dir.is_dir() else []
    need_te = not cached
    te_reason = ("%d cached conditioning blob(s) found" % len(cached) if cached
                 else "no cached conditioning blob")

rows, errors = [], []
live = LiveBox("02", "02 . models - running", kind="run")
state = {"label": "checking local caches", "started": time.perf_counter(), "expected": 0}


def _incomplete_bytes():
    total = 0
    for p in MODELS.rglob("*.incomplete"):
        try:
            total += p.stat().st_size
        except Exception:
            pass
    return total


def _tick02(live):
    live.set(rows=list(rows),
             note="%s · %s of %s downloaded · elapsed %s" % (
                 state["label"], gb(_incomplete_bytes()), gb(state["expected"]),
                 fmt_secs(time.perf_counter() - state["started"])))


ticker = Ticker(live, _tick02, 2.0).start()


def ensure_hf(label, repo, filename, local_dir, expected_bytes, revision=None):
    dest = Path(local_dir) / filename
    if dest.is_file() and dest.stat().st_size == expected_bytes and not FORCE_REDOWNLOAD:
        rows.append((label, "%s  cached" % gb(expected_bytes)))
        return dest
    state["label"] = "downloading " + label
    state["started"] = time.perf_counter()
    state["expected"] = expected_bytes
    try:
        hf_hub_download(repo_id=repo, filename=filename, revision=revision,
                        local_dir=str(local_dir), token=HF_TOKEN)
    except Exception as exc:
        errors.append("%s: %s: %s" % (label, type(exc).__name__, str(exc)[:220]))
        rows.append((label, "FAILED - see note"))
        return None
    if not dest.is_file() or dest.stat().st_size != expected_bytes:
        got = dest.stat().st_size if dest.is_file() else 0
        errors.append("%s: size mismatch (got %s, expected %s)"
                      % (label, got, expected_bytes))
        rows.append((label, "SIZE MISMATCH"))
        return None
    rows.append((label, "%s  downloaded" % gb(expected_bytes)))
    return dest

dit = DIT_CHOICES[DIT_CHOICE]
ensure_hf("DiT " + DIT_CHOICE, dit["repo"], dit["file"],
          MODELS / "diffusion_models", dit["bytes"])

WANTED_LORAS = {"v4 only": ["v4_step600_ema"],
                "v4 + v1": ["v4_step600_ema", "v1_850_ema"],
                "none": []}[TURBO_LORAS]
for choice in WANTED_LORAS:
    spec = TURBO_LORA_CHOICES[choice]
    ensure_hf("Turbo LoRA " + choice, spec["repo"], spec["file"],
              MODELS / "loras", spec["bytes"])
if not WANTED_LORAS:
    rows.append(("turbo LoRAs", "skipped (none)"))

ensure_hf("video VAE", MODEL_REPO, VAE_FILES["video"]["path"], MODELS,
          VAE_FILES["video"]["bytes"], revision=MODEL_REVISION)
ensure_hf("audio VAE", MODEL_REPO, VAE_FILES["audio"]["path"], MODELS,
          VAE_FILES["audio"]["bytes"], revision=MODEL_REVISION)

if need_te:
    ensure_hf("text encoder", MODEL_REPO, TE_FILE["path"], MODELS,
              TE_FILE["bytes"], revision=MODEL_REVISION)
else:
    rows.append(("text encoder", "skipped (%s)" % te_reason))

all_present = (MODELS / "diffusion_models" / dit["file"]).is_file() \
    and (MODELS / VAE_FILES["video"]["path"]).is_file() \
    and (MODELS / VAE_FILES["audio"]["path"]).is_file() \
    and all((MODELS / "loras" / TURBO_LORA_CHOICES[c]["file"]).is_file()
            for c in WANTED_LORAS) \
    and (not need_te or (MODELS / TE_FILE["path"]).is_file())
H3["models_ok"] = all_present

ticker.stop_now()
if errors:
    live.finish("02 . models - incomplete", rows=rows, kind="err",
                note="<br>".join(errors) + "<br>Downloads resume on rerun. If the repo is "
                     "gated, set a HF_TOKEN in Colab secrets.")
elif need_te:
    live.finish("02 . models ready (text encoder included)", rows=rows, kind="ok",
                note="One-time 14.6 GiB download. The Studio's Encode writes a conditioning "
                     "blob into /content/h3lab; it lasts only until the runtime recycles, so "
                     "later sessions need the text encoder again unless that blob survives.")
else:
    live.finish("02 . models ready", rows=rows, kind="ok",
                note="Text encoder skipped: %s. The Studio's Encode reuses the cached blob."
                     % te_reason)


In [ ]:
# @title 03 . studio { display-mode: "form" }
# ComfyUI backend + H3 Studio web UI (H3-Studio repo) behind a cloudflared tunnel.
# The cell prints only the tunnel URL and keeps running; stop it to shut everything down.
if "H3" not in globals():
    raise RuntimeError("run cell 00 first")
if "DIT_CHOICES" not in globals():
    raise RuntimeError("run cell 02 first (model catalog)")
if not (COMFY_DIR / ".git").is_dir():
    raise RuntimeError("ComfyUI is not installed; run cell 01 first")

import hashlib, json, math, re, shutil, socket, sys, threading, time, uuid
from pathlib import Path

import torch

try:
    import aiohttp  # noqa: F401
except ImportError:
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "aiohttp"], timeout=900)
    if rc:
        raise RuntimeError("aiohttp install failed: " + out[-400:])

STUDIO_PORT = 8000
REPO_URL = "https://github.com/GodL-x-SouL/H3-Studio.git"
REPO_DIR = WORK / "H3-Studio"
CLOUDFLARED = WORK / "cloudflared"
UPLOAD_DIR = WORK / "uploads"
LORA_DIR = MODELS / "loras"
LORA_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
FPS = 24
BASE_DIMS = {"0.3": (736, 416), "0.4": (864, 480)}
RATIOS = {"16:9": (16, 9), "9:16": (9, 16), "1:1": (1, 1), "4:3": (4, 3), "3:4": (3, 4)}
UNET_DTYPE_FLAGS = {"safe_fp16": "--force-fp16", "fp16_raw": "--force-fp16",
                    "bf16": "--bf16-unet", "fp16safe_node": None}
FIX_URL = ("https://raw.githubusercontent.com/Amduraznak/minimax-h3-fp16-fix/"
           "main/minimax_h3_fp16_fix.py")
SAGE_BASE = "https://github.com/netrunner-exe/SageAttention2-Colab-Wheels/raw/main"
SAGE_WHEELS = {
    ("cp312", "2.11.0", "128"): ("legacy_wheels", "colab-torch-cp312-2.11.0+cu128"),
    ("cp312", "2.11.0", "130"): ("legacy_wheels", "colab-torch-cp312-2.11.0+cu130"),
    ("cp312", "2.13.0", "130"): ("legacy_wheels", "colab-torch-cp312-2.13.0+cu130"),
    ("cp313", "2.11.0", "128"): ("optimized_wheels", "colab-torch-cp313-2.11.0+cu128"),
    ("cp313", "2.11.0", "130"): ("optimized_wheels", "colab-torch-cp313-2.11.0+cu130"),
    ("cp313", "2.13.0", "132"): ("optimized_wheels", "colab-torch-cp313-2.13.0+cu132"),
}
NODE_REPOS = {
    "fp16safe": ("FP16Safe", FP16SAFE_DIR,
                 "https://github.com/aaalll12322/ComfyUI-MiniMaxH3-FP16Safe.git"),
    "te_speed": ("TE-Speed", TESPEED_DIR,
                 "https://github.com/kaelzhang/TE-Speed-MiniMaxH3-MChip.git"),
    "spectrum": ("Spectrum", SPECTRUM_DIR,
                 "https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3.git"),
}


def _status(title, rows=None, kind="info", note="", details=""):
    return _box_html(title, rows or [], kind=kind, note=note, details=details)


def ratio_dims(gear, ratio):
    if ratio == "16:9":
        return BASE_DIMS[gear]
    budget = BASE_DIMS[gear][0] * BASE_DIMS[gear][1]
    rw, rh = RATIOS[ratio]
    width = max(32, round(math.sqrt(budget * rw / rh) / 32) * 32)
    height = max(32, round(math.sqrt(budget * rh / rw) / 32) * 32)
    return width, height


def _frame_sig(value):
    value = (value or "").strip()
    if not value:
        return None
    local = Path(value)
    if local.is_file():
        return hashlib.sha256(local.read_bytes()).hexdigest()
    return hashlib.sha256(("literal:" + value).encode("utf-8")).hexdigest()


def build_cfg(prompt, gear, ratio, length, steps, seed, dit_choice, lora_name, lora_strength,
              dtype_mode, use_sage, sage_mode, speed_method, te_mode, text_branch,
              budget_min, triton, extra_flags, first_frame, last_frame):
    prompt = (prompt or "").strip()
    if not prompt:
        raise ValueError("prompt is empty")
    length = int(length)
    if length < 5 or (length - 5) % 17 != 0:
        raise ValueError("length must satisfy n %% 17 == 5 (22, 39, ..., 124, ...); got %d"
                         % length)
    if gear not in BASE_DIMS or ratio not in RATIOS:
        raise ValueError("unknown gear/ratio %s/%s" % (gear, ratio))
    if dit_choice not in DIT_CHOICES:
        raise ValueError("unknown DiT choice %s" % dit_choice)
    width, height = ratio_dims(gear, ratio)
    steps = int(steps)
    lora_name = (lora_name or "off").strip()
    lora, lora_is_turbo = None, False
    if lora_name != "off":
        lora_path = LORA_DIR / lora_name
        if not lora_path.is_file():
            raise ValueError("LoRA not found: %s (download it in the web UI)" % lora_path)
        lora_is_turbo = "turbo" in lora_name.lower()
        lora = {"file": lora_name, "bytes": lora_path.stat().st_size}
    dtype_flag = UNET_DTYPE_FLAGS[dtype_mode]
    warnings = []
    use_sage_effective = bool(use_sage)
    if use_sage and dtype_flag != "--force-fp16":
        warnings.append("SageAttention needs fp16 activations; with %s the T4 falls back "
                        "to fp32, so Sage is disabled for this run. Use safe_fp16."
                        % dtype_mode)
        use_sage_effective = False
    if dtype_mode == "bf16":
        warnings.append("bf16 on T4 runs as fp32 compute: the slow baseline, ~342 s/step "
                        "at 864x480. Kept for comparison only.")
    if dtype_mode == "fp16_raw":
        warnings.append("fp16_raw is expected to produce NaN on H3 (condition_proj "
                        "overflow). Reference runs only.")
    if dtype_mode == "fp16safe_node" and lora is not None:
        warnings.append("fp16safe_node dequant-casts GGUF weights at runtime; watch "
                        "ComfyUI RSS in the Console tab. safe_fp16 remains the tested path.")
    if speed_method == "te_speed" and te_mode == "4-step LoRA" and steps not in (4, 5):
        warnings.append("TE-Speed '4-step LoRA' profile is tuned for 4 steps; steps=%d "
                        "may degrade output." % steps)
    if speed_method == "te_speed" and te_mode == "8-step LoRA" and steps != 8:
        warnings.append("TE-Speed '8-step LoRA' profile is tuned for 8 steps; steps=%d "
                        "may degrade output." % steps)
    if speed_method == "spectrum" and steps <= 6:
        warnings.append("Spectrum's win shrinks at low step counts; at %d steps TE-Speed "
                        "is the better fit." % steps)
    if lora_is_turbo and "v4" in lora_name and steps <= 4:
        warnings.append("v4 at <=4 steps can smear heavy motion; the v1 turbo LoRA is the "
                        "safer 4-step choice.")
    if dtype_mode != "bf16" and dit_choice == "abiray_Q4_K_M" and gear == "0.4" \
            and length > 56:
        warnings.append("Q4_K_M (10.77 GiB) + fp16-compute fp32 transients exceeds the "
                        "T4's ~14.5 GiB at the 0.4 gear with long clips (verified OOM at "
                        "864x480/124f). Use abiray_Q3_K_M, gear 0.3, or a shorter length.")
    if lora_is_turbo and steps > 10:
        warnings.append("Turbo LoRA is tuned for few steps; %d steps on top of it can "
                        "over-denoise. Choose LoRA 'off' for high step counts." % steps)
    if lora is None and steps <= 8:
        warnings.append("No LoRA: the base model usually wants more steps (12-20) than "
                        "the few-step LoRA profiles. steps=%d may look rough." % steps)
    if text_branch and dtype_mode == "safe_fp16":
        warnings.append("fp32 text branch is redundant with safe_fp16: the built-in fix "
                        "already upcasts condition_proj's input. Keep it off unless "
                        "diagnosing.")
    if speed_method == "te_speed" and te_mode != "Standard" and lora is None:
        warnings.append("TE-Speed '%s' profile is tuned around the turbo LoRA; without a "
                        "LoRA pick TE_MODE=Standard." % te_mode)
    first_frame = (first_frame or "").strip()
    last_frame = (last_frame or "").strip()
    first_sig, last_sig = _frame_sig(first_frame), _frame_sig(last_frame)
    key_payload = {
        "repo": MODEL_REPO, "revision": MODEL_REVISION,
        "text_encoder_sha256": TE_FILE["sha256"],
        "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
        "width": width, "height": height, "length": length,
    }
    if first_sig:
        key_payload["first_frame"] = first_sig
    if last_sig:
        key_payload["last_frame"] = last_sig
    key = "cond-" + hashlib.sha256(json.dumps(
        key_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()[:32]
    method_tag = []
    if use_sage_effective:
        method_tag.append("sage")
    if speed_method != "none":
        method_tag.append(speed_method)
    label = "%sstep_%dx%d_%df_%s%s" % (steps, width, height, length, dtype_mode,
                                       ("_" + "_".join(method_tag)) if method_tag else "")
    chain = "UnetLoaderGGUF"
    if lora is not None:
        chain += " -> TurboLoRA(%s@%s)" % (lora_name, lora_strength)
    if text_branch:
        chain += " -> Fp32TextBranch"
    if dtype_mode == "fp16safe_node":
        chain += " -> FP16Safe"
    if speed_method == "te_speed":
        chain += " -> TE-Speed(%s)" % te_mode
    if speed_method == "spectrum":
        chain += " -> Spectrum"
    if use_sage_effective:
        chain += " -> Sage(%s)" % sage_mode
    if lora is not None:
        chain += " -> %s" % ("MiniMaxH3TurboSampler" if lora_is_turbo
                             else "KSamplerSelect(euler)")
    cfg = {
        "prompt": prompt, "width": width, "height": height, "length": length, "fps": FPS,
        "gear": gear, "ratio": ratio, "steps": steps, "seed": int(seed),
        "dit": DIT_CHOICES[dit_choice], "dit_choice": dit_choice,
        "lora": lora, "lora_choice": lora_name, "lora_strength": float(lora_strength),
        "lora_is_turbo": lora_is_turbo, "lora_low_vram": False,
        "dtype_mode": dtype_mode, "dtype_flag": dtype_flag,
        "use_sage": use_sage_effective, "sage_mode": sage_mode,
        "speed_method": speed_method, "te_mode": te_mode,
        "text_branch": bool(text_branch), "triton": bool(triton),
        "extra_flags": (extra_flags or "").strip(),
        "first_frame": first_frame, "last_frame": last_frame,
        "budget_s": int(budget_min) * 60, "window_s": 120, "max_windows": 2,
        "sampler_start_timeout_s": 900,
        "key": key, "label": label, "latent_key": label,
        "video_prefix": "h3_lab/" + label, "expected_shapes": [
            [1, 24, ((length - 5) // 17) * 5 + 2, height // 16, width // 16],
            [1, 32, 2, round(length / FPS * 40)]],
        "te_file": TE_FILE, "video_vae": VAE_FILES["video"], "audio_vae": VAE_FILES["audio"],
        "blob_paths": [str(WORK / "conditioning" / (key + ".pt"))],
        "chain": chain, "warnings": warnings,
    }
    return cfg


def _payload_to_cfg(data):
    frames = int(data.get("frames") or 124)
    frames = max(22, min(277, ((frames - 5) // 17) * 17 + 5))
    scale = int(data.get("scale") or 100)
    gear = "0.4" if scale >= 80 else "0.3"
    speed = {"none": "none", "4-step": "none", "TE-Speed": "te_speed",
             "Spectrum": "spectrum"}.get(str(data.get("speed") or "none"), "none")
    return build_cfg(
        str(data.get("prompt") or ""), gear, str(data.get("ratio") or "16:9"), frames,
        int(data.get("steps") or 6), int(data.get("seed") or 0),
        str(data.get("dit_choice") or "abiray_Q3_K_M"),
        str(data.get("lora") or "off"), float(data.get("strength") or 1.0),
        str(data.get("dtype") or "safe_fp16"), bool(data.get("sage", True)),
        str(data.get("sage_mode") or "sageattn_qk_int8_pv_fp16_cuda"), speed,
        str(data.get("te_mode") or "4-step LoRA"), False,
        float(data.get("budget") or 75), bool(data.get("triton", True)),
        str(data.get("extra_flags") or ""), str(data.get("first_frame") or ""),
        str(data.get("last_frame") or ""))


def _lora_choices():
    return ["off"] + sorted(p.name for p in LORA_DIR.glob("*.safetensors"))


def _default_lora():
    preferred = TURBO_LORA_CHOICES["v4_step600_ema"]["file"]
    return preferred if (LORA_DIR / preferred).is_file() else "off"


def download_lora(url):
    url = (url or "").strip()
    if not url:
        raise ValueError("paste a HuggingFace or direct .safetensors URL first")
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
                     timeout=900)
        if rc:
            raise RuntimeError("huggingface_hub install failed: " + out[-400:])
        from huggingface_hub import hf_hub_download
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    target = None
    match = re.match(
        r"https?://huggingface\.co/([^/]+/[^/]+)/(?:blob|resolve)/([^/]+)/(.+)", url)
    if match:
        repo_id, revision, filename = match.groups()
        filename = filename.split("?")[0].strip()
        if not filename.endswith(".safetensors"):
            raise ValueError("only .safetensors LoRAs are supported")
        target = Path(hf_hub_download(repo_id=repo_id, filename=filename,
                                      revision=revision, local_dir=str(LORA_DIR),
                                      token=token))
    elif url.startswith(("http://", "https://")) and ".safetensors" in url:
        name = url.split("?")[0].rstrip("/").split("/")[-1]
        if not name.endswith(".safetensors"):
            raise ValueError("direct links must end in .safetensors")
        target = LORA_DIR / name
        if not target.is_file() or target.stat().st_size < 1_000_000:
            with requests.get(url, stream=True, timeout=1800) as response:
                response.raise_for_status()
                staged = target.with_suffix(".part")
                with staged.open("wb") as handle:
                    for chunk in response.iter_content(1024 * 1024):
                        handle.write(chunk)
                staged.replace(target)
    else:
        raise ValueError("expected https://huggingface.co/<repo>/blob/<rev>/<file>"
                         ".safetensors")
    if target.parent != LORA_DIR:
        moved = LORA_DIR / target.name
        shutil.copy2(target, moved)
        target = moved
    return target.name


def _download_fp16_fix():
    response = requests.get(FIX_URL, timeout=60)
    response.raise_for_status()
    text = response.text
    for needle in ("K_OUT_PROJ", "K_FC2", "condition_proj"):
        if needle not in text:
            raise RuntimeError("downloaded file does not look like the fp16 fix "
                               "(missing %s)" % needle)
    FP16_FIX_FILE.write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode()).hexdigest()


def _ensure_fp16_fix(mode):
    if mode == "safe_fp16":
        if not FP16_FIX_FILE.is_file():
            digest = _download_fp16_fix()
            H3["fp16_patch"] = True
            return "downloaded + active (sha %s)" % digest[:12]
        H3["fp16_patch"] = True
        return "active"
    existed = FP16_FIX_FILE.is_file()
    if existed:
        FP16_FIX_FILE.unlink()
    H3["fp16_patch"] = False
    return "removed for %s" % mode if existed else "off for %s" % mode


def sync_fp16_fix(active):
    if active:
        digest = _download_fp16_fix()
        H3["fp16_patch"] = True
        return "active (sha %s)" % digest[:12]
    existed = FP16_FIX_FILE.is_file()
    if existed:
        FP16_FIX_FILE.unlink()
    H3["fp16_patch"] = False
    return "removed" if existed else "not installed"


def _verify_node(kind):
    if kind == "fp16safe":
        path = FP16SAFE_DIR / "nodes.py"
        return path.is_file() and "MiniMaxH3FP16Safe" in path.read_text(
            encoding="utf-8", errors="ignore")
    if kind == "te_speed":
        path = TESPEED_DIR / "__init__.py"
        return path.is_file() and "TESpeedMiniMaxH3" in path.read_text(
            encoding="utf-8", errors="ignore")
    for path in SPECTRUM_DIR.rglob("*.py"):
        try:
            if "SpectrumApplyMiniMaxH3" in path.read_text(encoding="utf-8",
                                                          errors="ignore"):
                return True
        except Exception:
            pass
    return False


def sync_node(kind, active):
    if not active:
        return "off"
    label, dirpath, url = NODE_REPOS[kind]
    if not (dirpath / ".git").is_dir():
        rc, out = sh(["git", "clone", "--depth", "1", url, str(dirpath)], timeout=600)
        if rc:
            raise RuntimeError("%s clone failed: %s" % (label, out[-400:]))
    req = dirpath / "requirements.txt"
    if req.is_file():
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
                     timeout=900)
        if rc:
            raise RuntimeError("%s deps failed: %s" % (label, out[-400:]))
    if not _verify_node(kind):
        raise RuntimeError("%s node class not found after clone; repo layout changed"
                           % label)
    return "installed"


def setup_state():
    return {
        "fp16fix": FP16_FIX_FILE.is_file(),
        "fp16safe": (FP16SAFE_DIR / ".git").is_dir(),
        "tespeed": (TESPEED_DIR / ".git").is_dir(),
        "spectrum": (SPECTRUM_DIR / ".git").is_dir(),
        "sage": bool(H3.get("sage_ok")),
    }


def apply_setup(flags):
    flags = flags or {}
    rows = []
    for label, action in (
            ("fp16 NaN fix", lambda: sync_fp16_fix(bool(flags.get("fp16fix")))),
            ("FP16Safe", lambda: sync_node("fp16safe", bool(flags.get("fp16safe")))),
            ("TE-Speed", lambda: sync_node("te_speed", bool(flags.get("tespeed")))),
            ("Spectrum", lambda: sync_node("spectrum", bool(flags.get("spectrum"))))):
        try:
            rows.append((label, action()))
        except Exception as exc:
            rows.append((label, "FAILED: %s" % str(exc)[:200]))
    rows.append(("SageAttention", "import ok" if H3.get("sage_ok") else "not installed"))
    return rows


def install_sage():
    py_tag = "cp%d%d" % (sys.version_info.major, sys.version_info.minor)
    torch_tag = torch.__version__.split("+")[0].split(".dev")[0]
    cuda_tag = (torch.version.cuda or "").replace(".", "")
    key = (py_tag, torch_tag, cuda_tag)
    rows = [("runtime", "python %s, torch %s, cuda %s"
             % (py_tag, torch_tag, cuda_tag))]
    if key not in SAGE_WHEELS:
        rows.append(("state", "no prebuilt wheel for this runtime"))
        return rows
    subdir, folder = SAGE_WHEELS[key]
    filename = "sageattention-2.1.2-%s-%s-linux_x86_64.whl" % (py_tag, py_tag)
    url = "%s/%s/%s/%s" % (SAGE_BASE, subdir, folder, filename)
    wheel = WHEEL_DIR / filename
    try:
        if not wheel.is_file() or wheel.stat().st_size < 1_000_000:
            response = requests.get(url, timeout=600)
            response.raise_for_status()
            wheel.write_bytes(response.content)
        rows.append(("wheel", "%s (%.1f MiB)" % (filename, wheel.stat().st_size / 2**20)))
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                      str(wheel)], timeout=600)
        if rc:
            raise RuntimeError("pip install failed: " + out[-400:])
        rc, out = sh([sys.executable, "-c",
                      "import sageattention; print(getattr(sageattention, "
                      "'__version__', '?'))"])
        if rc:
            raise RuntimeError("import check failed: " + out[-400:])
        H3["sage_ok"] = True
        rows.append(("sageattention", out.strip()))
    except Exception as exc:
        H3["sage_ok"] = False
        rows.append(("state", "FAILED: %s" % str(exc)[:200]))
    return rows


try:
    import sageattention as _sage_probe
    H3["sage_ok"] = True
except Exception:
    H3["sage_ok"] = False
H3.setdefault("fp16_patch", FP16_FIX_FILE.is_file())
H3.setdefault("studio_last_ok", None)


def _sniff_ext(data):
    if data[:8] == b"\x89PNG\r\n\x1a\n":
        return ".png"
    if data[:2] == b"\xff\xd8":
        return ".jpg"
    if data[:4] == b"RIFF" and data[8:12] == b"WEBP":
        return ".webp"
    if data[:2] == b"BM":
        return ".bmp"
    return ".png"


def prepare_image(value, tag):
    value = (value or "").strip()
    if not value:
        return None
    src = Path(value)
    if src.is_file():
        data = src.read_bytes()
    elif value.lower().startswith(("http://", "https://")):
        response = requests.get(value, timeout=180)
        response.raise_for_status()
        data = response.content
    else:
        raise RuntimeError("%s_frame not found: %s (upload it in the Frames tab)"
                           % (tag, value))
    if len(data) < 512:
        raise RuntimeError("%s_frame image is suspiciously small (%d bytes)"
                           % (tag, len(data)))
    name = "h3lab_%s_%s%s" % (tag, hashlib.sha256(data).hexdigest()[:12], _sniff_ext(data))
    input_dir = COMFY_DIR / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    (input_dir / name).write_bytes(data)
    return name


def _stage_upload(filename, data):
    data = bytes(data)
    if len(data) < 512:
        raise RuntimeError("uploaded image is suspiciously small (%d bytes)" % len(data))
    ext = _sniff_ext(data)
    dest = UPLOAD_DIR / ("h3lab_%s%s" % (hashlib.sha256(data).hexdigest()[:12], ext))
    if not dest.is_file():
        dest.write_bytes(data)
    return str(dest)


def ensure_server(cfg, force=False):
    want = cfg["dtype_flag"]
    extra = (["--enable-triton-backend"] if cfg["triton"] else []) \
        + (cfg["extra_flags"].split() if cfg["extra_flags"] else [])
    stats = None
    try:
        stats = requests.get(BASE_URL + "/system_stats", timeout=5).json()
    except Exception:
        stats = None
    if stats is not None and not force:
        argv = stats.get("system", {}).get("argv") or []
        mismatch = False
        if want and want not in argv:
            mismatch = True
        if want is None and ("--force-fp16" in argv or "--bf16-unet" in argv):
            mismatch = True
        if cfg["triton"] and "--enable-triton-backend" not in argv:
            mismatch = True
        if not mismatch:
            offset = max(0, COMFY_LOG.stat().st_size - 20000) if COMFY_LOG.exists() else 0
            return stats, offset
    return start_comfy(want, extra=extra)


def _log_tail_text():
    if not COMFY_LOG.exists():
        return "(no log yet)"
    text = COMFY_LOG.read_bytes()[-60000:].decode("utf-8", errors="replace")
    lines = [l for l in text.splitlines() if l.strip()]
    return "\n".join(lines[-150:])


def op_encode(cfg, keep_server):
    key = cfg["key"]
    store = WORK / "conditioning"
    store.mkdir(parents=True, exist_ok=True)
    dest = store / (key + ".pt")
    if dest.is_file() and dest.stat().st_size > 1024 * 1024:
        H3["blob_ready"] = True
        H3["studio_last_ok"] = True
        yield (_status("Encode recalled", [
            ("blob", str(dest)), ("size", gb(dest.stat().st_size)), ("key", key)],
            kind="ok",
            note="Conditioning for this prompt/geometry already exists; nothing to encode."),
            "", None)
        return
    te_path = MODELS / cfg["te_file"]["path"]
    if not te_path.is_file():
        H3["studio_last_ok"] = False
        yield (_status("Encode blocked", [
            ("text encoder", "missing at %s" % te_path),
            ("fix", "run cell 02 with the text encoder included")], kind="err"),
            "", None)
        return
    rows, errors, log_lines = [("key", key), ("geometry", "%dx%d / %d frames"
                                              % (cfg["width"], cfg["height"],
                                                 cfg["length"]))], [], []
    started = time.perf_counter()
    try:
        image_names = {}
        for tag, slot in (("first", "first_frame"), ("last", "last_frame")):
            name = prepare_image(cfg.get(slot, ""), tag)
            if name:
                image_names[tag] = name
        if image_names:
            rows.append(("images", ", ".join("%s_frame=%s" % kv
                                             for kv in sorted(image_names.items()))))
        yield (_status("Encode", rows, kind="run",
                       note="starting ComfyUI / staging inputs ..."), "", None)
        stats, log_offset = ensure_server(cfg, force=not keep_server)
        info = get_object_info()
        H3["object_info"] = info
        for node in (["CLIPLoader", "VAELoader", "MiniMaxH3ImageToVideo",
                      "H3SaveConditioning"] + (["LoadImage"] if image_names else [])):
            if node not in info:
                raise RuntimeError("%s missing from object_info; rerun cell 01" % node)
        node_spec = info["MiniMaxH3ImageToVideo"].get("input", {})
        node_inputs = set(node_spec.get("required", {})) | set(node_spec.get("optional", {}))
        for tag, slot in (("first", "first_frame"), ("last", "last_frame")):
            if tag in image_names and slot not in node_inputs:
                raise RuntimeError("this ComfyUI build's MiniMaxH3ImageToVideo has no %s "
                                   "input (schema: %s)" % (slot, sorted(node_inputs)))
        graph = {
            "2": {"class_type": "CLIPLoader", "inputs": {
                "clip_name": cfg["te_file"]["name"], "type": "minimax",
                "device": "default"}},
            "3": {"class_type": "VAELoader",
                  "inputs": {"vae_name": cfg["video_vae"]["name"]}},
            "5": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {
                "clip": ["2", 0], "vae": ["3", 0], "prompt": cfg["prompt"],
                "width": cfg["width"], "height": cfg["height"],
                "length": cfg["length"]}},
            "40": {"class_type": "H3SaveConditioning", "inputs": {
                "conditioning": ["5", 0], "samples": ["5", 1],
                "store_dir": str(store), "key": key}},
        }
        if "first" in image_names:
            graph["0"] = {"class_type": "LoadImage",
                          "inputs": {"image": image_names["first"]}}
            graph["5"]["inputs"]["first_frame"] = ["0", 0]
        if "last" in image_names:
            graph["00"] = {"class_type": "LoadImage",
                           "inputs": {"image": image_names["last"]}}
            graph["5"]["inputs"]["last_frame"] = ["00", 0]
        problems = validate_graph(graph, info)
        if problems:
            raise RuntimeError("encode graph invalid:\n" + "\n".join(problems))
        prompt_id = str(uuid.uuid4())
        shown = log_offset
        response = requests.post(BASE_URL + "/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        if response.json().get("node_errors"):
            raise RuntimeError("Node errors: %s" % response.json()["node_errors"])
        record = None
        while time.perf_counter() - started < 3600:
            proc = H3.get("proc")
            if proc is not None and proc.poll() is not None:
                raise RuntimeError("ComfyUI exited during encode (OOM kill?)")
            chunk = tail_comfy_log(shown, 500)
            shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
            if chunk:
                log_lines.extend(chunk.splitlines())
            progress = latest_progress(chunk)
            if progress:
                status_line = "%s  %d/%d" % (
                    pbar(progress["steps_completed"] / max(1, progress["steps_total"])),
                    progress["steps_completed"], progress["steps_total"])
            else:
                lines = [l for l in chunk.splitlines() if l.strip()]
                status_line = (lines[-1][:150].replace("<", "&lt;") if lines
                               else "(initializing)")
            yield (_status("Encode", rows, kind="run",
                           note="elapsed %s<br>%s" % (fmt_secs(time.perf_counter()
                                                                - started), status_line)),
                   "\n".join(log_lines[-200:]), None)
            try:
                history = requests.get(BASE_URL + "/history/" + prompt_id,
                                       timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True \
                            or status.get("status_str") == "error":
                        record = history[prompt_id]
                        break
            except Exception:
                pass
            time.sleep(3)
        if record is None:
            raise TimeoutError("encode did not finish within 3600s")
        if record.get("status", {}).get("status_str") != "success":
            raise RuntimeError("encode failed: %s" % record.get("status"))
        rows += [("status", "encoded in %.0fs" % (time.perf_counter() - started)),
                 ("blob", "%s (%s)" % (dest, gb(dest.stat().st_size)))]
        summary_path = store / (key + ".summary.json")
        if summary_path.is_file():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            rows.append(("shapes", json.dumps(
                summary.get("samples", {}).get("parts", []))[:120].replace("\\n", " ")))
        H3["blob_ready"] = True
    except Exception as exc:
        errors.append("%s: %s" % (type(exc).__name__, str(exc)[:400]))
    finally:
        if not keep_server:
            stop_comfy()
    if errors:
        H3["studio_last_ok"] = False
        yield (_status("Encode FAILED", rows, kind="err", note="<br>".join(errors)),
               "\n".join(log_lines[-200:]), None)
    else:
        H3["studio_last_ok"] = True
        yield (_status("Encode done", rows, kind="ok",
                       note="ComfyUI stopped to free RAM." if not keep_server
                       else "ComfyUI left running for the next step."),
               "\n".join(log_lines[-200:]), None)


def op_sample(cfg, keep_server):
    key = cfg["key"]
    latent_dir = WORK / "latents"
    latent_dir.mkdir(parents=True, exist_ok=True)
    cond_blob = WORK / "conditioning" / (key + ".pt")
    if not cond_blob.is_file():
        H3["studio_last_ok"] = False
        yield (_status("Sample blocked", [("conditioning", "missing: %s" % cond_blob),
                                          ("fix", "run Encode first")], kind="err"),
               "", None)
        return
    if cfg["use_sage"] and not H3.get("sage_ok"):
        H3["studio_last_ok"] = False
        yield (_status("Sample blocked", [
            ("sage", "wheel not installed"),
            ("fix", "Setup tab: Install SageAttention wheel, or turn Use Sage off")],
            kind="err"), "", None)
        return
    rows, errors, warnings, log_lines = [], [], [], []
    stop_watch = None
    started = time.perf_counter()
    try:
        rows.append(("fp16 fix", _ensure_fp16_fix(cfg["dtype_mode"])))
        stats, log_offset = ensure_server(cfg, force=not keep_server)
        info = get_object_info()
        H3["object_info"] = info
        needed = {"UnetLoaderGGUF", "H3LoadConditioning", "RandomNoise", "BasicScheduler",
                  "BasicGuider", "SamplerCustomAdvanced", "KSamplerSelect", "H3SaveLatent",
                  "H3LatentProbe"}
        if cfg["lora"]:
            needed.add("MiniMaxH3TurboLoRA")
            if cfg["lora_is_turbo"]:
                needed.add("MiniMaxH3TurboSampler")
        if cfg["dtype_mode"] == "fp16safe_node":
            needed.add("MiniMaxH3FP16Safe")
        if cfg["speed_method"] == "te_speed":
            needed.add("TESpeedMiniMaxH3")
        if cfg["speed_method"] == "spectrum":
            needed.add("SpectrumApplyMiniMaxH3")
        missing = sorted(n for n in needed if n not in info)
        if missing:
            raise RuntimeError("nodes not loaded: %s - install them in the Setup tab, then "
                               "run again" % ", ".join(missing))
        model_src = ["1", 0]
        extra_nodes = {}
        if cfg["lora"]:
            extra_nodes["19"] = {"class_type": "MiniMaxH3TurboLoRA", "inputs": {
                "model": model_src, "lora_name": cfg["lora"]["file"],
                "strength": cfg["lora_strength"], "low_vram": cfg["lora_low_vram"]}}
            model_src = ["19", 0]
        if cfg["text_branch"]:
            extra_nodes["17"] = {"class_type": "H3Fp32TextBranch",
                                 "inputs": {"model": model_src}}
            model_src = ["17", 0]
        if cfg["dtype_mode"] == "fp16safe_node":
            extra_nodes["14"] = {"class_type": "MiniMaxH3FP16Safe",
                                 "inputs": {"model": model_src}}
            model_src = ["14", 0]
        if cfg["speed_method"] == "te_speed":
            extra_nodes["21"] = {"class_type": "TESpeedMiniMaxH3", "inputs": {
                "model": model_src, "processing_control_value": 0.12,
                "processing_percent_1": 0.1, "processing_percent_2": 0.9, "mcs": 2,
                "device": "auto", "mode": cfg["te_mode"], "cache_depth": -1.0,
                "verbose": False}}
            model_src = ["21", 0]
        if cfg["speed_method"] == "spectrum":
            extra_nodes["22"] = {"class_type": "SpectrumApplyMiniMaxH3", "inputs": {
                "model": model_src, "enabled": True, "blend_weight": 0.5, "degree": 1,
                "ridge_lambda": 0.1, "window_size": 2.0, "flex_window": 0.75,
                "warmup_steps": 1, "tail_actual_steps": 1, "max_history": 8,
                "debug": False}}
            model_src = ["22", 0]
        if cfg["use_sage"]:
            extra_nodes["20"] = {"class_type": "H3SagePatch",
                                 "inputs": {"model": model_src,
                                            "mode": cfg["sage_mode"]}}
            model_src = ["20", 0]
        graph = {
            "1": {"class_type": "UnetLoaderGGUF",
                  "inputs": {"unet_name": cfg["dit"]["file"]}},
            "41": {"class_type": "H3LoadConditioning", "inputs": {
                "store_dir": str(WORK / "conditioning"), "key": key}},
            "6": {"class_type": "RandomNoise", "inputs": {"noise_seed": cfg["seed"]}},
            "7": {"class_type": "BasicScheduler", "inputs": {
                "model": model_src, "scheduler": "simple", "steps": cfg["steps"],
                "denoise": 1.0}},
            "8": ({"class_type": "MiniMaxH3TurboSampler", "inputs": {}}
                  if (cfg["lora"] and cfg["lora_is_turbo"])
                  else {"class_type": "KSamplerSelect",
                        "inputs": {"sampler_name": "euler"}}),
            "9": {"class_type": "BasicGuider", "inputs": {
                "model": model_src, "conditioning": ["41", 0]}},
            "10": {"class_type": "SamplerCustomAdvanced", "inputs": {
                "noise": ["6", 0], "guider": ["9", 0], "sampler": ["8", 0],
                "sigmas": ["7", 0], "latent_image": ["41", 1]}},
            "18": {"class_type": "H3SaveLatent", "inputs": {
                "samples": ["10", 0], "store_dir": str(latent_dir),
                "key": cfg["latent_key"]}},
            "15": {"class_type": "H3LatentProbe", "inputs": {
                "samples": ["18", 0], "filename_prefix": cfg["video_prefix"]}},
        }
        graph.update(extra_nodes)
        problems = validate_graph(graph, info)
        if problems:
            raise RuntimeError("sample graph invalid: " + "; ".join(problems))
        (WORK / "sample_graph.json").write_text(json.dumps(graph, indent=2),
                                                encoding="utf-8")
        rows += [("graph", ", ".join(sorted(n["class_type"]
                                            for n in graph.values()))[:200]),
                 ("sampler", "MiniMaxH3TurboSampler"
                  if (cfg["lora"] and cfg["lora_is_turbo"]) else "KSamplerSelect (euler)")]
        prompt_id = str(uuid.uuid4())
        shown = log_offset
        maxima = {"rss_gib": 0.0, "vram_mib": 0.0, "host_low_gib": 1e9}
        stop_watch = threading.Event()

        def watch():
            tick = 0
            while not stop_watch.is_set():
                try:
                    host = psutil.virtual_memory().available / GIB
                    maxima["host_low_gib"] = min(maxima["host_low_gib"], host)
                    proc = H3.get("proc")
                    if proc is not None:
                        try:
                            rss = psutil.Process(proc.pid).memory_info().rss
                            for child in psutil.Process(proc.pid).children(recursive=True):
                                try:
                                    rss += child.memory_info().rss
                                except Exception:
                                    pass
                            maxima["rss_gib"] = max(maxima["rss_gib"], rss / GIB)
                        except Exception:
                            pass
                    if tick % 2 == 0:
                        rc, out = sh(["nvidia-smi", "--query-gpu=memory.used",
                                      "--format=csv,noheader,nounits"], timeout=10)
                        if rc == 0:
                            try:
                                maxima["vram_mib"] = max(
                                    maxima["vram_mib"],
                                    float(out.strip().splitlines()[0]))
                            except Exception:
                                pass
                except Exception:
                    pass
                tick += 1
                stop_watch.wait(1.0)

        watcher = threading.Thread(target=watch, daemon=True)
        watcher.start()

        def cancel(pid):
            try:
                requests.post("%s/api/jobs/%s/cancel" % (BASE_URL, pid), timeout=30)
            except Exception:
                try:
                    requests.post(BASE_URL + "/interrupt", timeout=30)
                except Exception:
                    pass

        window_s, budget_s = cfg["window_s"], cfg["budget_s"]
        guard = {"outcome": None, "windows": 0, "sampler_started_at": None,
                 "settled": False, "projected": None, "observed_s_per_step": None}
        response = requests.post(BASE_URL + "/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        if response.json().get("node_errors"):
            raise RuntimeError("Node errors: %s" % response.json()["node_errors"])
        rows.append(("submitted", "%d steps @ %dx%d / %d frames"
                     % (cfg["steps"], cfg["width"], cfg["height"], cfg["length"])))
        deadline = started + cfg["sampler_start_timeout_s"] + budget_s + 1800
        record = None
        while time.perf_counter() < deadline:
            proc = H3.get("proc")
            if proc is not None and proc.poll() is not None:
                raise RuntimeError("ComfyUI exited during sampling (OOM kill?)")
            new = tail_comfy_log(shown, 500)
            shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
            if new:
                log_lines.extend(new.splitlines())
            try:
                history = requests.get(BASE_URL + "/history/" + prompt_id,
                                       timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True \
                            or status.get("status_str") == "error":
                        record = history[prompt_id]
                        break
            except Exception:
                pass
            now = time.perf_counter()
            progress = latest_progress("\n".join(log_lines[-400:]))
            if progress is None:
                yield (_status("Sampling", rows, kind="run",
                               note="loading model / waiting for the sampler to start · "
                                    "elapsed %s" % fmt_secs(now - started)),
                       "\n".join(log_lines[-200:]), None)
                if now - started > cfg["sampler_start_timeout_s"]:
                    guard["outcome"] = "SAMPLER_NEVER_STARTED"
                    cancel(prompt_id)
                    break
            else:
                if guard["sampler_started_at"] is None:
                    guard["sampler_started_at"] = now - progress["elapsed_in_loop_s"]
                observed = now - guard["sampler_started_at"]
                if not guard["settled"]:
                    if progress["steps_completed"] <= 0:
                        no_step_deadline = max(window_s * cfg["max_windows"],
                                               2.0 * budget_s / max(cfg["steps"], 1))
                        if observed >= no_step_deadline:
                            guard["outcome"] = "NO_STEP_COMPLETED"
                            cancel(prompt_id)
                            break
                    elif observed >= window_s * (guard["windows"] + 1):
                        guard["windows"] += 1
                        guard["observed_s_per_step"] = progress["observed_s_per_step"]
                        guard["projected"] = progress["observed_s_per_step"] * cfg["steps"]
                        if guard["projected"] > budget_s:
                            guard["outcome"] = "PROJECTED_OVER_BUDGET"
                            cancel(prompt_id)
                            break
                        guard["settled"] = True
                s_per_step = progress["observed_s_per_step"]
                remaining = (s_per_step * (cfg["steps"] - progress["steps_completed"])
                             if s_per_step else None)
                live_rows = rows + [
                    ("sampling", "%s  step %d/%d" % (
                        pbar(progress["steps_completed"]
                             / max(1, progress["steps_total"])),
                        progress["steps_completed"], progress["steps_total"])),
                    ("s/step", ("%.1f" % s_per_step) if s_per_step else "measuring ..."),
                    ("eta", fmt_secs(remaining) if remaining else "?"),
                    ("sampling elapsed", fmt_secs(observed)),
                    ("VRAM", "%.0f MiB" % maxima["vram_mib"]),
                    ("guard", "%d/%d windows" % (guard["windows"], cfg["max_windows"])),
                ]
                yield (_status("Sampling", live_rows, kind="run",
                               note="working · elapsed %s total"
                                    % fmt_secs(now - started)),
                       "\n".join(log_lines[-200:]), None)
            time.sleep(3)
        else:
            guard["outcome"] = "OVERALL_TIMEOUT"
            cancel(prompt_id)
        stop_watch.set()
        watcher.join(timeout=5)
        status_str = (record or {}).get("status", {}).get("status_str")
        if guard["outcome"] is not None:
            status = "CANCELLED_" + guard["outcome"]
        elif record is None:
            status = "NO_HISTORY"
        else:
            status = "SUCCESS" if status_str == "success" else "FAILED"
        full_log = "\n".join(log_lines)
        final_progress = latest_progress(full_log)
        if final_progress and final_progress.get("observed_s_per_step"):
            rows.append(("s/step", "%.1f (last completed step)"
                         % final_progress["observed_s_per_step"]))
        if guard["projected"]:
            rows.append(("projected denoise", "%.1f min for %d steps"
                         % (guard["projected"] / 60, cfg["steps"])))
        rows += [
            ("status", status + (" / %s" % status_str if status_str else "")),
            ("elapsed", "%.0f s" % (time.perf_counter() - started)),
            ("peak VRAM", "%.0f MiB" % maxima["vram_mib"]),
            ("peak ComfyUI RSS", "%.2f GiB" % maxima["rss_gib"]),
            ("min host free", "%.2f GiB" % maxima["host_low_gib"]),
        ]
        if guard["outcome"]:
            warnings.append("Guard stopped the run: %s" % guard["outcome"])
        saved = {}
        for line in reversed(log_lines):
            if "H3_SAVE_LATENT " in line:
                try:
                    saved = json.loads(line.split("H3_SAVE_LATENT ", 1)[1])
                except Exception:
                    pass
                break
        if saved:
            rows.append(("latent", "%s (%s)" % (saved.get("path"),
                                                gb(saved.get("bytes", 0)))))
        summary_file = OUTPUT_DIR / (cfg["video_prefix"] + ".latent_summary.json")
        if summary_file.is_file():
            summary = json.loads(summary_file.read_text(encoding="utf-8"))
            finite = summary.get("finite")
            rows.append(("latent finite", str(finite)))
            if finite is not None and not all(finite):
                warnings.append("Sampled latent is NOT finite - this run is a NaN result. "
                                "On newer ComfyUI also try Extra flags="
                                "--disable-dynamic-vram (known fp32 NaN trigger).")
            rows.append(("latent shapes", "match" if summary.get("shapes")
                         == cfg["expected_shapes"] else "MISMATCH %s"
                         % summary.get("shapes")))
        if cfg["lora"]:
            turbo = [l for l in log_lines if "[MiniMaxH3TurboLoRA]" in l]
            rows.append(("lora", turbo[-1][:120] if turbo
                         else "NO LOG - lora may not have applied"))
        if cfg["speed_method"] == "te_speed":
            te = [l for l in log_lines if "[te_speed]" in l]
            rows.append(("te-speed", te[-1][:120] if te else "NO LOG - did not engage"))
            if not te:
                warnings.append("TE-Speed never logged; it likely did not engage (check "
                                "the Setup tab and restart).")
        if cfg["use_sage"]:
            sage = [l for l in log_lines if "sage attention override active" in l]
            rows.append(("sage", sage[-1][:120] if sage else "NO LOG"))
            if not sage:
                warnings.append("Sage override never logged; attention ran the default "
                                "path.")
        if cfg["text_branch"]:
            tb = [l for l in log_lines if "H3_FP32_TEXT_BRANCH" in l]
            rows.append(("fp32 text branch", tb[-1][:120] if tb else "NO LOG"))
        if cfg["dtype_mode"] == "safe_fp16" \
                and re.search(r"manual cast: torch\.float32", full_log):
            warnings.append("Log still says 'manual cast: torch.float32' - the fp16 fix "
                            "did NOT take effect; check the Setup tab.")
        if "Unexpected architecture type" in full_log:
            warnings.append("GGUF architecture patch did not take - rerun cell 01.")
        import_failures = [l.strip() for l in log_lines
                           if "IMPORT FAILED" in l or "failed to import" in l.lower()]
        if import_failures:
            warnings.append("Import failures: " + "; ".join(import_failures[:3]))
        if re.search(r"CUDA out of memory|torch\.OutOfMemoryError", full_log, re.I):
            hint = ""
            if cfg["dit_choice"] == "abiray_Q4_K_M" and cfg["dtype_mode"] != "bf16" \
                    and cfg["gear"] == "0.4":
                hint = (" Q4_K_M + fp16 compute exceeds the T4's ~14.5 GiB at this "
                        "geometry; switch to abiray_Q3_K_M (re-encode is not needed).")
            warnings.append("CUDA OOM seen in the log." + hint)
        H3["last_latent_key"] = cfg["latent_key"]
        metrics = {"status": status, "guard": guard,
                   "s_per_step": final_progress.get("observed_s_per_step")
                   if final_progress else None,
                   "maxima": maxima, "saved_latent": saved, "label": cfg["label"],
                   "steps": cfg["steps"], "dtype_mode": cfg["dtype_mode"],
                   "speed_method": cfg["speed_method"], "sage": cfg["use_sage"]}
        (WORK / ("metrics_%s.json" % cfg["label"])).write_text(
            json.dumps(metrics, indent=2), encoding="utf-8")
    except Exception as exc:
        errors.append("%s: %s" % (type(exc).__name__, str(exc)[:400]))
    finally:
        if stop_watch is not None:
            stop_watch.set()
        sample_log_text = "\n".join(log_lines) if log_lines else tail_comfy_log(
            max(0, COMFY_LOG.stat().st_size - 60000) if COMFY_LOG.exists() else 0, 80)
        (WORK / ("sample_log_%s.txt" % cfg["label"])).write_text(sample_log_text,
                                                                 encoding="utf-8")
        if not keep_server:
            stop_comfy()
    if errors:
        H3["studio_last_ok"] = False
        yield (_status("Sample FAILED", rows, kind="err", note="<br>".join(errors),
                       details=sample_log_text[-5000:]),
               sample_log_text[-5000:], None)
    else:
        status_lookup = next((v for k, v in rows if k == "status"), "")
        H3["studio_last_ok"] = status_lookup.startswith("SUCCESS")
        kind = "ok" if H3["studio_last_ok"] else (
            "warn" if status_lookup.startswith("CANCELLED") else "err")
        yield (_status("Sample " + status_lookup.split(" / ")[0], rows, kind=kind,
                       note="<br>".join(warnings) if warnings else "Decode next.",
                       details=sample_log_text[-5000:] if warnings else ""),
               sample_log_text[-5000:], None)


def op_decode(cfg, keep_server=False):
    latent_path = WORK / "latents" / (cfg["latent_key"] + ".pt")
    summary_file = OUTPUT_DIR / (cfg["video_prefix"] + ".latent_summary.json")
    finite = None
    if summary_file.is_file():
        try:
            finite = json.loads(summary_file.read_text(encoding="utf-8")).get("finite")
        except Exception:
            pass
    if not latent_path.is_file():
        H3["studio_last_ok"] = False
        yield (_status("Decode idle", [("latent", "missing: %s" % latent_path),
                                       ("fix", "run Sample first")], kind="idle"),
               "", None)
        return
    if finite is not None and not all(finite):
        H3["studio_last_ok"] = False
        yield (_status("Decode refused", [("latent", str(latent_path)),
                                          ("finite", str(finite))], kind="warn",
                       note="The latent is non-finite (NaN run); decoding would produce "
                            "noise. Rerun Sample with a working dtype mode."),
               "", None)
        return
    rows, errors, log_lines = [], [], []
    video_file = None
    inspection = {}
    started = time.perf_counter()
    try:
        stop_comfy()
        start_comfy(None)
        info = get_object_info()
        H3["object_info"] = info

        def decode_fill(node_type, known):
            spec_in = info[node_type].get("input", {})
            required = spec_in.get("required") or {}
            optional = spec_in.get("optional") or {}
            inputs, unresolved = {}, []
            for name, value in known.items():
                if name in optional and name not in required:
                    inputs[name] = value
            unknown = sorted(set(known) - set(required) - set(optional))
            if unknown:
                raise RuntimeError("%s does not accept %s" % (node_type, unknown))
            for name, spec in required.items():
                if name in known:
                    inputs[name] = known[name]
                    continue
                options = spec[0] if spec else None
                config = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
                dynamic = config.get("options") \
                    if options == "COMFY_DYNAMICCOMBO_V3" else None
                if dynamic:
                    def needs_nothing(option):
                        return not ((option.get("inputs") or {}).get("required") or {})

                    choice = next((o for o in dynamic if o.get("key") == "auto"
                                   and needs_nothing(o)),
                                  next((o for o in dynamic if needs_nothing(o)), None))
                    if choice is None:
                        unresolved.append("%s: dynamic combo has no free option" % name)
                    else:
                        inputs[name] = choice["key"]
                elif isinstance(options, list) and options:
                    inputs[name] = config.get("default", options[0])
                elif "default" in config:
                    inputs[name] = config["default"]
                else:
                    unresolved.append("%s: %s" % (name, spec))
            if unresolved:
                raise RuntimeError("%s requires inputs with no value here: %s"
                                   % (node_type, "; ".join(unresolved)))
            return inputs

        graph = {
            "1": {"class_type": "VAELoader", "inputs": decode_fill(
                "VAELoader", {"vae_name": cfg["video_vae"]["name"]})},
            "2": {"class_type": "VAELoader", "inputs": decode_fill(
                "VAELoader", {"vae_name": cfg["audio_vae"]["name"]})},
            "3": {"class_type": "H3LoadLatent", "inputs": decode_fill(
                "H3LoadLatent", {"store_dir": str(WORK / "latents"),
                                 "key": cfg["latent_key"]})},
            "4": {"class_type": "VAEDecode", "inputs": decode_fill(
                "VAEDecode", {"samples": ["3", 0], "vae": ["1", 0]})},
            "5": {"class_type": "VAEDecodeAudio", "inputs": decode_fill(
                "VAEDecodeAudio", {"samples": ["3", 0], "vae": ["2", 0]})},
            "6": {"class_type": "CreateVideo", "inputs": decode_fill(
                "CreateVideo", {"images": ["4", 0], "audio": ["5", 0],
                                "fps": float(cfg["fps"])})},
            "7": {"class_type": "SaveVideo", "inputs": decode_fill(
                "SaveVideo", {"video": ["6", 0], "filename_prefix": cfg["video_prefix"]})},
        }
        rows.append(("decode graph", ", ".join("%s:%s" % (i, s["class_type"])
                                               for i, s in sorted(graph.items()))))
        yield (_status("Decode", rows, kind="run",
                       note="starting ComfyUI without the dtype flag (VAE-only RAM) ..."),
               "", None)
        prompt_id = str(uuid.uuid4())
        log_offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
        response = requests.post(BASE_URL + "/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        if response.json().get("node_errors"):
            raise RuntimeError("Node errors: %s" % response.json()["node_errors"])
        record, shown = None, log_offset
        while time.perf_counter() - started < 1800:
            proc = H3.get("proc")
            if proc is not None and proc.poll() is not None:
                raise RuntimeError("ComfyUI exited during decode (OOM kill?)")
            new = tail_comfy_log(shown, 400)
            shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
            if new:
                log_lines.extend(new.splitlines())
            tail_line = ([l for l in new.splitlines() if l.strip()][-1][:150]
                         .replace("<", "&lt;") if new and new.strip()
                         else (log_lines[-1][:150].replace("<", "&lt;")
                               if log_lines else "(initializing)"))
            yield (_status("Decode", rows, kind="run",
                           note="decoding · elapsed %s<br>%s"
                                % (fmt_secs(time.perf_counter() - started),
                                   tail_line)),
                   "\n".join(log_lines[-200:]), None)
            try:
                history = requests.get(BASE_URL + "/history/" + prompt_id,
                                       timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True \
                            or status.get("status_str") == "error":
                        record = history[prompt_id]
                        break
            except Exception:
                pass
            time.sleep(3)
        if record is None:
            raise TimeoutError("decode did not finish within 1800s")
        if record.get("status", {}).get("status_str") != "success":
            raise RuntimeError("decode failed: %s" % record.get("status"))
        rows.append(("status", "decoded in %.0fs" % (time.perf_counter() - started)))
        produced = []
        for node_output in (record.get("outputs") or {}).values():
            for entry in (node_output.get("images", []) + node_output.get("videos", [])
                          + node_output.get("gifs", [])):
                if isinstance(entry, dict) and entry.get("filename"):
                    produced.append(OUTPUT_DIR / entry.get("subfolder", "")
                                    / entry["filename"])
        if not produced:
            directory = OUTPUT_DIR / "h3_lab"
            produced = sorted([p for p in directory.glob(cfg["label"] + "*")
                               if p.suffix in (".mp4", ".webm")],
                              key=lambda p: p.stat().st_mtime) \
                if directory.is_dir() else []
        produced = [p for p in produced if p.is_file()]
        if not produced:
            raise RuntimeError("SaveVideo wrote nothing findable in %s" % OUTPUT_DIR)
        video_file = produced[-1]
        rows.append(("file", "%s (%.1f MiB)" % (video_file,
                                                video_file.stat().st_size / 2**20)))
        try:
            import av
            with av.open(str(video_file)) as container:
                streams = {s.type: s for s in container.streams}
                inspection["streams"] = sorted(streams)
                if "video" in streams:
                    vs = streams["video"]
                    inspection["video"] = {
                        "codec": vs.codec_context.name,
                        "size": "%dx%d" % (vs.codec_context.width,
                                           vs.codec_context.height),
                        "duration_s": round(float(vs.duration * vs.time_base), 2)
                        if vs.duration else None}
                if "audio" in streams:
                    aus = streams["audio"]
                    inspection["audio"] = {
                        "codec": aus.codec_context.name,
                        "rate": aus.codec_context.sample_rate,
                        "channels": aus.codec_context.channels}
                brightness = []
                index = 0
                for frame in container.decode(video=0):
                    if index % 8 == 0:
                        plane = frame.to_ndarray(format="gray")
                        brightness.append((float(plane.mean()), float(plane.std())))
                    index += 1
                inspection["frames_decoded"] = index
                if brightness:
                    means = [m for m, _ in brightness]
                    stds = [s for _, s in brightness]
                    inspection["mean_luma"] = round(sum(means) / len(means), 2)
                    inspection["near_black"] = sum(1 for m, s in zip(means, stds)
                                                   if m < 4.0 and s < 2.0)
                    inspection["sampled"] = len(means)
        except Exception as exc:
            inspection["error"] = "%s: %s" % (type(exc).__name__, str(exc)[:200])
        rows.append(("inspection", json.dumps(inspection)[:220]))
    except Exception as exc:
        errors.append("%s: %s" % (type(exc).__name__, str(exc)[:400]))
    finally:
        if not keep_server:
            stop_comfy()
    if errors:
        H3["studio_last_ok"] = False
        yield (_status("Decode FAILED", rows, kind="err", note="<br>".join(errors),
                       details="\n".join(log_lines)[-4000:]),
               "\n".join(log_lines[-200:]), None)
    else:
        H3["studio_last_ok"] = True
        note = "Saved under ComfyUI/output and served by the web UI."
        if inspection.get("near_black") and inspection.get("sampled") \
                and inspection["near_black"] == inspection["sampled"]:
            note += " Every sampled frame is dark and flat - the file may be black."
        yield (_status("Decode done", rows, kind="ok", note=note),
               "\n".join(log_lines[-200:]), str(video_file))


def run_job(payload, update, cancel_event):
    cfg = _payload_to_cfg(payload)
    H3["cfg"] = cfg
    H3["studio_last_ok"] = None
    update(label=cfg["label"], dims="%dx%d" % (cfg["width"], cfg["height"]),
           frames=cfg["length"], steps=cfg["steps"], seed=cfg["seed"],
           warnings=list(cfg["warnings"]))

    def drive(gen, stage, lo, hi):
        last = None
        for _status_html, log_text, extra in gen:
            if cancel_event.is_set():
                try:
                    requests.post(BASE_URL + "/interrupt", timeout=15)
                except Exception:
                    pass
                raise RuntimeError("cancelled by user")
            progress = None
            found = latest_progress(log_text or "")
            if found:
                progress = lo + (hi - lo) * found["percent"] / 100.0
            update(stage=stage, progress=progress, log_tail=(log_text or "")[-4000:])
            if extra:
                last = extra
        return last

    try:
        drive(op_encode(cfg, keep_server=True), "encode", 0.0, 0.08)
        if not H3.get("studio_last_ok"):
            raise RuntimeError("encode failed — check the Console tab")
        drive(op_sample(cfg, keep_server=True), "sampling", 0.08, 0.9)
        if not H3.get("studio_last_ok"):
            raise RuntimeError(H3.get("sample_error")
                               or "sampling failed — check the Console tab")
        path = drive(op_decode(cfg, keep_server=True), "decode", 0.9, 1.0)
        if not H3.get("studio_last_ok"):
            raise RuntimeError("decode failed — check the Console tab")
    except Exception:
        try:
            stop_comfy()
        except Exception:
            pass
        raise
    video_rel = None
    if path:
        try:
            video_rel = Path(str(path)).relative_to(OUTPUT_DIR).as_posix()
        except Exception:
            video_rel = None
    return {
        "video": ("/h3/output?file=%s" % video_rel) if video_rel else None,
        "meta": {"dims": "%dx%d" % (cfg["width"], cfg["height"]),
                 "frames": cfg["length"]},
        "warnings": list(cfg["warnings"]),
    }


def _pick_port(start=STUDIO_PORT):
    for port in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                probe.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise RuntimeError("no free port in %d..%d" % (start, start + 49))


def _free_port(port):
    freed = []
    try:
        for conn in psutil.net_connections(kind="inet"):
            try:
                if getattr(conn, "status", None) != psutil.CONN_LISTEN:
                    continue
                if not conn.laddr or conn.laddr.port != port or not conn.pid:
                    continue
                os.kill(conn.pid, signal.SIGTERM)
                freed.append(conn.pid)
            except Exception:
                pass
    except Exception:
        pass
    if freed:
        time.sleep(1.0)
    for pid in list(freed):
        try:
            if psutil.pid_exists(pid):
                os.kill(pid, signal.SIGKILL)
        except Exception:
            pass
    return freed


def _ensure_repo():
    if (REPO_DIR / ".git").is_dir():
        rc, out = sh(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                      "origin", "main"], timeout=600)
        if rc == 0:
            sh(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], timeout=120)
        else:
            print("repo update skipped: %s" % out.strip().splitlines()[-1][:160]
                  if out.strip() else "repo update skipped")
    else:
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        rc, out = sh(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                     timeout=900)
        if rc:
            raise RuntimeError("H3-Studio clone failed: %s" % out[-400:])
    if not (REPO_DIR / "dist" / "index.html").is_file():
        raise RuntimeError("H3-Studio checkout has no dist/ (build missing?)")
    if not (REPO_DIR / "server" / "h3_backend.py").is_file():
        raise RuntimeError("H3-Studio checkout has no server/h3_backend.py")


def _ensure_cloudflared():
    if CLOUDFLARED.is_file() and CLOUDFLARED.stat().st_size > 10_000_000:
        return
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
           "cloudflared-linux-amd64")
    response = requests.get(url, timeout=600)
    response.raise_for_status()
    CLOUDFLARED.write_bytes(response.content)
    CLOUDFLARED.chmod(0o755)


def _start_tunnel(port, timeout_s=90):
    proc = subprocess.Popen(
        [str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:%d" % port,
         "--protocol", "http2", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True)
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    found = {}

    def reader():
        try:
            for line in proc.stdout:
                if not found.get("url"):
                    match = pattern.search(line)
                    if match:
                        found["url"] = match.group(0)
                found.setdefault("tail", [])
                found["tail"] = (found["tail"] + [line.strip()])[-8:]
        except Exception:
            pass

    threading.Thread(target=reader, daemon=True, name="cloudflared-reader").start()
    started = time.time()
    while time.time() - started < timeout_s:
        if found.get("url"):
            return proc, found["url"]
        if proc.poll() is not None:
            raise RuntimeError("cloudflared exited early: %s"
                               % " | ".join(found.get("tail") or [])[-300:])
        time.sleep(0.5)
    try:
        proc.terminate()
    except Exception:
        pass
    raise RuntimeError("cloudflared printed no URL within %ds" % timeout_s)


def _teardown(tunnel_proc, backend_handle, port):
    if tunnel_proc is not None:
        try:
            os.killpg(tunnel_proc.pid, signal.SIGTERM)
            tunnel_proc.wait(timeout=8)
        except Exception:
            try:
                os.killpg(tunnel_proc.pid, signal.SIGKILL)
            except Exception:
                pass
    if backend_handle is not None:
        try:
            backend_handle.stop()
        except Exception:
            pass
    try:
        stop_comfy()
    except Exception:
        pass
    _free_port(port)
    _free_port(8188)


# --- bring everything up -----------------------------------------------------

_ensure_repo()
sys.path.insert(0, str(REPO_DIR / "server"))
sys.modules.pop("h3_backend", None)
import h3_backend  # noqa: E402

print("ComfyUI starting ...")
try:
    start_comfy("--force-fp16", wait_s=900)
    print("ComfyUI up on 127.0.0.1:8188")
except Exception as exc:
    print("ComfyUI prestart failed (jobs retry on demand): %s" % str(exc)[:200])

studio_port = _pick_port()
backend = h3_backend.start(port=studio_port, hooks={
    "dist_dir": str(REPO_DIR / "dist"),
    "comfy_url": BASE_URL,
    "output_dir": str(OUTPUT_DIR),
    "make_job": run_job,
    "list_loras": _lora_choices,
    "download_lora": download_lora,
    "log_tail": _log_tail_text,
    "setup_state": setup_state,
    "apply_setup": apply_setup,
    "install_sage": install_sage,
    "stage_upload": _stage_upload,
})
print("web ui + api on 127.0.0.1:%d" % studio_port)

tunnel_proc, tunnel_url, tunnel_error = None, None, ""
try:
    _ensure_cloudflared()
    tunnel_proc, tunnel_url = _start_tunnel(studio_port)
except Exception as exc:
    tunnel_error = "%s: %s" % (type(exc).__name__, str(exc)[:300])

if tunnel_url:
    studio_live = LiveBox("studio", "03 . studio live", rows=[
        ("url", '<a href="%s" target="_blank" style="color:#1a5fb4">%s</a>'
         % (tunnel_url, tunnel_url)),
    ], kind="ok",
        note="Open the URL in a new tab — that is the whole Studio (ComfyUI tunneled "
             "through the same host). Stopping this cell closes the tunnel, the web "
             "server and ComfyUI.")
else:
    studio_live = LiveBox("studio", "03 . studio running (tunnel unavailable)", rows=[
        ("local", "http://127.0.0.1:%d (reachable inside this VM only)" % studio_port),
        ("tunnel", tunnel_error or "not started"),
    ], kind="warn",
        note="The web UI and ComfyUI are up; only the public tunnel failed. Stop this "
             "cell and rerun it to retry.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    pass
finally:
    _teardown(tunnel_proc, backend, studio_port)

studio_live.finish("03 . studio stopped", rows=[
    ("tunnel", "closed"),
    ("web ui", "stopped"),
    ("ComfyUI", "stopped"),
], kind="idle",
    note="Ports %d and 8188 freed. Rerun this cell to start again." % studio_port)
